In [ ]:
# Terminal + Ollama configuration with one-switch profiles for T4
import os

# Choose one profile: "agente", "codigo", "geral", "imagem"
PROFILE = "geral"
CONTEXT_LENGTH = 16384

PROFILE_PRESETS = {
    "agente": {
        "default_model": "nous-hermes2:10.7b",
        "preload_models": ["nous-hermes2:10.7b"],
        "max_loaded_models": 1,
        "num_parallel_requests": 1
    },
    "codigo": {
        "default_model": "qwen2.5-coder:7b",
        "preload_models": ["qwen2.5-coder:7b", "deepseek-coder:6.7b"],
        "max_loaded_models": 2,
        "num_parallel_requests": 2
    },
    "geral": {
        "default_model": "llama3.1:8b",
        "preload_models": ["llama3.1:8b", "mistral:7b"],
        "max_loaded_models": 2,
        "num_parallel_requests": 2
    },
    "imagem": {
        "default_model": "llama3.1:8b",
        "preload_models": ["llama3.1:8b"],
        "max_loaded_models": 1,
        "num_parallel_requests": 1
    }
}

if PROFILE not in PROFILE_PRESETS:
    raise ValueError(f"Invalid PROFILE: {PROFILE}. Use one of: {list(PROFILE_PRESETS)}")

preset = PROFILE_PRESETS[PROFILE]
DEFAULT_MODEL = preset["default_model"]
PRELOAD_MODELS = preset["preload_models"]
MAX_LOADED_MODELS = preset["max_loaded_models"]
NUM_PARALLEL_REQUESTS = preset["num_parallel_requests"]

TERMINAL_PORT = 7681
OLLAMA_PORT = 11434

os.environ["OLLAMA_CONTEXT_LENGTH"] = str(CONTEXT_LENGTH)
os.environ["OLLAMA_HOST"] = "0.0.0.0"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = str(MAX_LOADED_MODELS)
os.environ["OLLAMA_NUM_PARALLEL"] = str(NUM_PARALLEL_REQUESTS)

print("Ollama env configured")
print(f"- PROFILE={PROFILE}")
print(f"- DEFAULT_MODEL={DEFAULT_MODEL}")
print(f"- PRELOAD_MODELS={PRELOAD_MODELS}")
print(f"- OLLAMA_CONTEXT_LENGTH={os.environ['OLLAMA_CONTEXT_LENGTH']}")
print(f"- OLLAMA_MAX_LOADED_MODELS={MAX_LOADED_MODELS}")
print(f"- OLLAMA_NUM_PARALLEL={NUM_PARALLEL_REQUESTS}")

## Fluxo simples (sem repeticao)

1. Execute a celula 1 para configurar variaveis (inclui `CONTEXT_LENGTH=16384`).
2. Execute a celula 3 **Simple one-click startup**.
3. Escolha profile/modelos e clique em **Start selected services**.
4. O Ollama sera sempre exposto publicamente; o terminal web e opcional (checkbox).

In [ ]:
# Simple one-click startup: always expose Ollama, optionally expose web terminal
# Run this cell, choose options, then click "Start selected services".

import os
import re
import sys
import time
import shutil
import signal
import subprocess
import threading
import requests

try:
    import ipywidgets as widgets
    from IPython.display import display
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])
    import ipywidgets as widgets
    from IPython.display import display

# Reuse existing globals when available
TERMINAL_PORT = globals().get("TERMINAL_PORT", 7681)
OLLAMA_PORT = globals().get("OLLAMA_PORT", 11434)
CONTEXT_LENGTH = int(globals().get("CONTEXT_LENGTH", 16384))
PROFILE = globals().get("PROFILE", "geral")
PROFILE_PRESETS = globals().get("PROFILE_PRESETS", {
    "agente": {
        "default_model": "nous-hermes2:10.7b",
        "preload_models": ["nous-hermes2:10.7b"]
    },
    "codigo": {
        "default_model": "qwen2.5-coder:7b",
        "preload_models": ["qwen2.5-coder:7b", "deepseek-coder:6.7b"]
    },
    "geral": {
        "default_model": "llama3.1:8b",
        "preload_models": ["llama3.1:8b", "mistral:7b"]
    },
    "imagem": {
        "default_model": "llama3.1:8b",
        "preload_models": ["llama3.1:8b"]
    }
})

# Keep process handlers between runs
if "SIMPLE_PROCS" not in globals():
    SIMPLE_PROCS = {
        "ollama_server": None,
        "ollama_tunnel": None,
        "ttyd": None,
        "terminal_tunnel": None,
    }


def sh(cmd):
    subprocess.check_call(["bash", "-lc", cmd])


def stop_proc(name):
    p = SIMPLE_PROCS.get(name)
    if p and p.poll() is None:
        p.terminate()
        try:
            p.wait(timeout=3)
        except subprocess.TimeoutExpired:
            p.kill()
    SIMPLE_PROCS[name] = None


def ensure_dependencies(enable_terminal):
    sh("apt-get update -y")
    sh("apt-get install -y curl wget")

    if shutil.which("ollama") is None:
        sh("sudo apt-get install -y zstd || true")
        sh("curl -fsSL https://ollama.com/install.sh | sh")

    # Keep cloudflared local to this notebook folder
    sh("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared")
    sh("chmod +x cloudflared")

    if enable_terminal and shutil.which("ttyd") is None:
        sh("apt-get install -y ttyd || true")
        sh("if [ ! -x /usr/bin/ttyd ]; then wget -q https://github.com/tsl0922/ttyd/releases/latest/download/ttyd.x86_64 -O /usr/local/bin/ttyd && chmod +x /usr/local/bin/ttyd; fi")


def wait_for_ollama(timeout=180):
    for i in range(timeout):
        try:
            r = requests.get(f"http://localhost:{OLLAMA_PORT}", timeout=2)
            if r.status_code in [200, 404]:
                print(f"Ollama is up after {i+1}s")
                return
        except requests.exceptions.RequestException:
            pass
        time.sleep(1)
    raise RuntimeError("Ollama did not start in time")


def start_ollama_server():
    os.environ["OLLAMA_CONTEXT_LENGTH"] = str(CONTEXT_LENGTH)
    os.environ["OLLAMA_HOST"] = "0.0.0.0"
    os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

    p = SIMPLE_PROCS.get("ollama_server")
    if p is not None and p.poll() is None:
        print("Ollama server already running")
    else:
        SIMPLE_PROCS["ollama_server"] = subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )
        print("Starting Ollama server...")
    wait_for_ollama()


def pull_and_warm_models(models):
    resolved = list(dict.fromkeys([m.strip() for m in models if m and m.strip()]))
    if not resolved:
        return []

    for model in resolved:
        print(f"Pulling {model}...")
        subprocess.check_call(["ollama", "pull", model])

    for model in resolved:
        payload = {
            "model": model,
            "prompt": "reply with: ready",
            "stream": False,
            "options": {
                "num_predict": 8,
                "num_ctx": CONTEXT_LENGTH
            },
            "keep_alive": "2h"
        }
        r = requests.post(f"http://localhost:{OLLAMA_PORT}/api/generate", json=payload, timeout=300)
        r.raise_for_status()
        print(f"Warm and running: {model}")

    return resolved


def start_tunnel(local_port, proc_key):
    stop_proc(proc_key)
    proc = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", f"http://localhost:{local_port}", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    SIMPLE_PROCS[proc_key] = proc

    start = time.time()
    while time.time() - start < 60:
        line = proc.stdout.readline()
        if not line:
            continue
        print(line.strip())
        m = re.search(r"(https://.*\.trycloudflare\.com)", line)
        if m:
            return m.group(1)

    raise RuntimeError(f"Could not create public URL for port {local_port}")


def parse_custom_models(raw_text):
    if not raw_text.strip():
        return []
    return [m.strip() for m in raw_text.split(",") if m.strip()]


available_models = sorted({
    m
    for cfg in PROFILE_PRESETS.values()
    for m in cfg.get("preload_models", [])
})
default_models = tuple(PROFILE_PRESETS.get(PROFILE, {}).get("preload_models", []))

profile_dropdown_simple = widgets.Dropdown(
    options=list(PROFILE_PRESETS.keys()),
    value=PROFILE if PROFILE in PROFILE_PRESETS else list(PROFILE_PRESETS.keys())[0],
    description="Profile:",
    layout=widgets.Layout(width="260px")
)

terminal_checkbox = widgets.Checkbox(
    value=True,
    description="Expose web terminal",
    indent=False
)

models_select = widgets.SelectMultiple(
    options=available_models,
    value=default_models,
    description="Models:",
    rows=min(8, max(4, len(available_models))),
    layout=widgets.Layout(width="420px")
)

custom_models_text = widgets.Text(
    value="",
    description="Custom:",
    placeholder="model1, model2",
    layout=widgets.Layout(width="520px")
)

start_button = widgets.Button(
    description="Start selected services",
    button_style="success",
    icon="play"
)

simple_output = widgets.Output()


def sync_models_with_profile(change):
    selected_profile = change["new"]
    preset_models = tuple(PROFILE_PRESETS.get(selected_profile, {}).get("preload_models", []))
    valid_models = [m for m in preset_models if m in available_models]
    models_select.value = tuple(valid_models)


profile_dropdown_simple.observe(sync_models_with_profile, names="value")


def on_start(_):
    with simple_output:
        simple_output.clear_output()
        try:
            selected_profile = profile_dropdown_simple.value
            selected_models = list(models_select.value)
            selected_models.extend(parse_custom_models(custom_models_text.value))

            if not selected_models:
                selected_models = PROFILE_PRESETS[selected_profile].get("preload_models", [])

            selected_models = list(dict.fromkeys(selected_models))

            globals()["PROFILE"] = selected_profile
            globals()["PRELOAD_MODELS"] = selected_models

            print("Preparing dependencies...")
            ensure_dependencies(enable_terminal=terminal_checkbox.value)

            print(f"Starting Ollama with context={CONTEXT_LENGTH}...")
            start_ollama_server()

            print("Pulling/warming selected models...")
            warmed = pull_and_warm_models(selected_models)

            print("Exposing Ollama API publicly...")
            ollama_url = start_tunnel(OLLAMA_PORT, "ollama_tunnel")

            terminal_url = None
            if terminal_checkbox.value:
                stop_proc("ttyd")
                SIMPLE_PROCS["ttyd"] = subprocess.Popen(
                    ["ttyd", "-p", str(TERMINAL_PORT), "bash", "-lc", "cd /content && exec bash"],
                    stdout=subprocess.PIPE,
                    stderr=subprocess.STDOUT,
                    text=True
                )
                print("Exposing web terminal publicly...")
                terminal_url = start_tunnel(TERMINAL_PORT, "terminal_tunnel")
            else:
                stop_proc("ttyd")
                stop_proc("terminal_tunnel")

            print("\nReady")
            print(f"- Profile: {selected_profile}")
            print(f"- Context: {CONTEXT_LENGTH}")
            print(f"- Models: {warmed if warmed else selected_models}")
            print(f"- Ollama URL: {ollama_url}")
            if terminal_url:
                print(f"- Terminal URL: {terminal_url}")

            globals()["ollama_url"] = ollama_url
            if terminal_url:
                globals()["terminal_url"] = terminal_url

        except Exception as e:
            print(f"Startup failed: {e}")


start_button.on_click(on_start)

display(widgets.VBox([
    widgets.HTML("<b>Simple Init</b>: choose profile/models and click once."),
    widgets.HBox([profile_dropdown_simple, terminal_checkbox]),
    models_select,
    custom_models_text,
    start_button,
    simple_output
]))

In [ ]:
# Model options and useful terminal commands for Google Colab T4
MODEL_OPTIONS = {
    "agente_geral_hermes": [
        "ollama pull nous-hermes2:10.7b",
        "ollama run nous-hermes2:10.7b"
    ],
    "codificacao": [
        "ollama pull qwen2.5-coder:7b",
        "ollama pull deepseek-coder:6.7b",
        "ollama run qwen2.5-coder:7b"
    ],
    "uso_geral": [
        "ollama pull llama3.1:8b",
        "ollama pull mistral:7b",
        "ollama run llama3.1:8b"
    ],
    "geracao_de_imagem": [
        "python3 -m pip install -q diffusers transformers accelerate safetensors",
        "python3 /content/generate_image_t4.py"
    ]
}

print(f"Active PROFILE: {PROFILE}")
print("\nChange PROFILE in the first cell to switch presets quickly.")
print("Available PROFILE values: agente, codigo, geral, imagem")

print("\nRecommended command options by use case:")
for category, commands in MODEL_OPTIONS.items():
    print(f"\n[{category}]")
    for cmd in commands:
        print(cmd)

print("\nRun multiple models in parallel on T4 (watch VRAM):")
print("ollama pull llama3.1:8b && ollama pull qwen2.5-coder:7b")
print("curl -s http://localhost:11434/api/generate -d '{\"model\":\"llama3.1:8b\",\"prompt\":\"ready\",\"stream\":false,\"keep_alive\":\"2h\"}'")
print("curl -s http://localhost:11434/api/generate -d '{\"model\":\"qwen2.5-coder:7b\",\"prompt\":\"ready\",\"stream\":false,\"keep_alive\":\"2h\"}'")
print("ollama ps")

In [ ]:
# Optional image model starter for T4 (save script and run from web terminal or notebook)
image_script = r'''
import torch
from diffusers import AutoPipelineForText2Image

model_id = "stabilityai/sdxl-turbo"
pipe = AutoPipelineForText2Image.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

prompt = "a cinematic cyberpunk city at sunset, ultra detailed, 4k"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
output_path = "/content/sdxl_t4_output.png"
image.save(output_path)
print(f"Image saved to {output_path}")
'''

with open("/content/generate_image_t4.py", "w", encoding="utf-8") as f:
    f.write(image_script)

print("Created: /content/generate_image_t4.py")
print("To run:")
print("python3 -m pip install -q diffusers transformers accelerate safetensors")
print("python3 /content/generate_image_t4.py")